# 🔬 Diagnostic Test Suite: PL FFT & Dual-DMA Pipeline Validation

This notebook performs step-by-step isolated unit testing on the **FPGA Hardware Trigger**, **Dual-DMA Engine**, **PL-accelerated FFT/CORDIC**, and **AD3 Signal Generator** without running the full dashboard UI.

--- 
## 1. System Permissions & USB Verification

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay
from pynq_oscilloscope.fft_dma import StreamingFFT
import numpy as np
import time
import matplotlib.pyplot as plt

# Grant permissions to access the AD3 USB bus
check_usb_permissions()

--- 
## 2. Load Overlay & Start AD3 Test Signal
Loads the `v1.2.0-rc1` bitstream, verifies both DMA blocks, and starts a **10 kHz Sine wave** (1.5V amplitude, 1.65V DC offset).

In [ ]:
# 1. Load Hardware Overlay
ol = OscilloscopeOverlay()
print("✅ Overlay loaded successfully!")
print(f"  • Time DMA (axi_dma_0) : {ol.xadc.dma}")
print(f"  • FFT DMA  (axi_dma_1) : {ol.fft.dma}")

# 2. Configure Hardware Trigger: Single-Shot Mode @ 1.65V
ol.trigger.configure(mode="Single", edge="Rising", threshold_volts=1.65, timeout_ms=50.0)
print(f"  • Hardware Trigger     : {ol.trigger}")

# 3. Start 10 kHz Sine Wave on AD3 (W1 -> A0)
ol.wavegen.start(shape="Sine", frequency=10000.0, amplitude=1.5, offset=1.65)
time.sleep(1.0)
print("✅ AD3 active at 10 kHz (1.5V amplitude, 1.65V offset).")

--- 
## 3. Test 1: Synchronized 10-Frame Arm-on-Demand Capture
Verifies that the Arm-on-Demand sequence (`transfer` $\rightarrow$ `arm` $\rightarrow$ `wait`) captures 10 consecutive frames synchronously with zero hangs or timeouts.

In [ ]:
print("🚀 Running Single-Shot Hardware-Synchronized Capture (10 frames)...")

ol.trigger.configure(mode="Single", edge="Rising", threshold_volts=1.65, timeout_ms=50.0)

for i in range(10):
    t0 = time.time()
    
    # 1. Prime BOTH DMA channels FIRST (Both DMAs now wait with TREADY=1)
    ol.xadc.dma.recvchannel.transfer(ol.xadc._buffer)
    ol.fft.dma.recvchannel.transfer(ol.fft._buffer)
    
    # 2. Arm the hardware trigger (Captures exactly 1 frame starting at Sample 0, then auto-disarms)
    ol.trigger.arm()
    
    # 3. Wait for hardware completion
    ol.xadc.dma.recvchannel.wait()
    ol.fft.dma.recvchannel.wait()
    
    dt = (time.time() - t0) * 1000.0
    
    # Process Time-Domain data
    raw_time = np.array(ol.xadc._buffer)
    voltages = (raw_time >> 4) * (3.3 / 4095.0)
    
    # Process Frequency-Domain data
    raw_fft = np.array(ol.fft._buffer, copy=True)
    raw_half = raw_fft[:1024].astype(np.float64)
    linear_volts = (raw_half / 2048.0) * (3.3 / 4095.0)
    mags = 20.0 * np.log10(np.maximum(linear_volts, 1e-6))
    
    peak_idx = np.argmax(mags[10:]) + 10 # Ignore DC bin
    peak_f = ol.fft.freq_axis[peak_idx]
    
    print(f"  Frame {i:02d}: Time={dt:5.1f} ms | Vpp={voltages.max()-voltages.min():.2f}V | Peak f0={peak_f/1e3:5.2f} kHz @ {mags[peak_idx]:.1f} dBV")
    time.sleep(0.02)

print("\n✅ All 10 frames completed with ZERO hangs or timeouts!")

--- 
## 4. Test 2: Hardware FPGA FFT vs. Software NumPy FFT Visual Comparison
Captures 1 frame and plots the raw FPGA CORDIC magnitude output against a reference software NumPy FFT to verify spectrum shape and peak alignment.

In [ ]:
# 1. Capture 1 clean frame
ol.xadc.dma.recvchannel.transfer(ol.xadc._buffer)
ol.fft.dma.recvchannel.transfer(ol.fft._buffer)
ol.trigger.arm()
ol.xadc.dma.recvchannel.wait()
ol.fft.dma.recvchannel.wait()

raw_time = np.array(ol.xadc._buffer)
voltages = (raw_time >> 4) * (3.3 / 4095.0)

raw_fft = np.array(ol.fft._buffer, copy=True)
raw_half = raw_fft[:1024].astype(np.float64)

# 2. Print first 15 raw FFT values
print("📊 First 15 Raw FFT Magnitude bins:")
for idx in range(15):
    freq_hz = ol.fft.freq_axis[idx]
    print(f"  Bin {idx:02d} ({freq_hz:6.1f} Hz): raw={raw_half[idx]:6.0f} (0x{int(raw_half[idx]):04X})")

# 3. Compute Software Reference FFT
voltages_ac = voltages - np.mean(voltages)
sw_fft_raw = np.abs(np.fft.rfft(voltages_ac))[:1024]

# 4. Plot Comparison
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), dpi=100)

ax1.plot(ol.fft.freq_axis[:100] / 1e3, raw_half[:100], color="#FF007F", linewidth=1.8, marker="o", markersize=3, label="Raw FPGA CORDIC Output")
ax1.set_title("Raw FPGA FFT Magnitude (Bins 0 to 100, 0 to 48.8 kHz)", fontweight="bold")
ax1.set_xlabel("Frequency (kHz)")
ax1.set_ylabel("Raw Integer Counts")
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

ax2.plot(ol.fft.freq_axis[:100] / 1e3, sw_fft_raw[:100], color="#00CCFF", linewidth=1.8, label="Software NumPy FFT (Reference)")
ax2.set_title("Software NumPy FFT Reference (0 to 48.8 kHz)", fontweight="bold")
ax2.set_xlabel("Frequency (kHz)")
ax2.set_ylabel("Linear Amplitude")
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

--- 
## 5. Test 3: Continuous 50-Frame Stress Test (Hardware Pipeline Throughput)
Captures 50 consecutive frames in pure Python to measure sustained frame latency (ms) and throughput (FPS).

In [ ]:
print("🚀 Running 50-Frame Continuous Streaming Stress Test...")
latencies = []

for frame_idx in range(50):
    t_start = time.time()
    try:
        # 1. Queue DMAs
        ol.xadc.dma.recvchannel.transfer(ol.xadc._buffer)
        ol.fft.dma.recvchannel.transfer(ol.fft._buffer)
        
        # 2. Arm trigger
        ol.trigger.arm()
        
        # 3. Wait
        ol.xadc.dma.recvchannel.wait()
        ol.fft.dma.recvchannel.wait()
        
        elapsed_ms = (time.time() - t_start) * 1000.0
        latencies.append(elapsed_ms)
        
        if frame_idx % 10 == 0:
            raw_time = np.array(ol.xadc._buffer)
            v = (raw_time >> 4) * (3.3 / 4095.0)
            raw_fft = np.array(ol.fft._buffer, copy=True)[:1024].astype(np.float64)
            mags = 20.0 * np.log10(np.maximum((raw_fft / 2048.0) * (3.3 / 4095.0), 1e-6))
            peak_idx = np.argmax(mags[10:]) + 10
            peak_f = ol.fft.freq_axis[peak_idx]
            print(f"  Frame {frame_idx:02d}: {elapsed_ms:4.1f} ms | Vpp: {v.max()-v.min():.2f}V | Peak f0: {peak_f/1e3:5.2f} kHz @ {mags[peak_idx]:.1f} dBV")
            
    except Exception as e:
        print(f"❌ Stalled at frame {frame_idx}: {e}")
        break

avg_time = np.mean(latencies)
fps = 1000.0 / avg_time
print(f"\n🎉 50/50 Frames Completed Successfully!")
print(f"  • Average frame latency: {avg_time:.2f} ms")
print(f"  • Sustained hardware throughput: ~{fps:.1f} FPS")

--- 
## 6. Test 4: Harmonic Distortion Test (Square Wave Odd Harmonics)
Switches AD3 to a 10 kHz Square wave to test Fourier odd harmonic resolution ($10\,\text{kHz}$, $30\,\text{kHz}$, $50\,\text{kHz}$, $70\,\text{kHz}$, $90\,\text{kHz}$).

In [ ]:
# 1. Switch AD3 to a 10 kHz Square wave
ol.wavegen.update_parameters(shape="Square", frequency=10000.0, amplitude=1.2)
time.sleep(0.5)

# 2. Capture frame
ol.xadc.dma.recvchannel.transfer(ol.xadc._buffer)
ol.fft.dma.recvchannel.transfer(ol.fft._buffer)
ol.trigger.arm()
ol.xadc.dma.recvchannel.wait()
ol.fft.dma.recvchannel.wait()

raw_fft = np.array(ol.fft._buffer, copy=True)[:1024].astype(np.float64)
mags_sq = 20.0 * np.log10(np.maximum((raw_fft / 2048.0) * (3.3 / 4095.0), 1e-6))

# 3. Plot Square Wave Spectrum
plt.figure(figsize=(10, 4), dpi=100)
plt.plot(ol.fft.freq_axis[:200] / 1e3, mags_sq[:200], color="#E040FB", linewidth=1.6, label="10 kHz Square Wave Spectrum")
plt.title("FPGA FFT: Square Wave Odd Harmonics (10k, 30k, 50k, 70k, 90k Hz)", fontweight="bold")
plt.xlabel("Frequency (kHz)")
plt.ylabel("Magnitude (dBV)")
plt.xlim(0, 100)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

--- 
## 7. Clean Hardware Shutdown

In [ ]:
ol.close()
print("🔒 Hardware handles closed and CMA memory released.")